# Airline Delay Propagation Analysis — 02: SQL Analysis (DuckDB)

This notebook loads `cleaned_flights.csv` (from `01_data_cleaning.ipynb`) into DuckDB and runs the full
SQL analysis layer, then exports `airline_delay_analysis.csv` for Power BI.

This dataset has a genuine `tailnum`, so propagation is scored using the **true** aircraft-rotation
`LAG()` methodology: for each flight, look up the same physical aircraft's previous-leg arrival delay.

DuckDB runs in-process inside Colab (`pip install duckdb`) and queries a Pandas DataFrame directly with
standard SQL -- no external database server required.


## 1. Install & Import

In [1]:
!pip install duckdb -q

import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

print(f"DuckDB version: {duckdb.__version__}")


DuckDB version: 1.3.2


## 2. Load Cleaned Data

In [17]:
try:
    flights = pd.read_csv('/content/cleaned_flights.csv', low_memory=False)
    print(f"Loaded cleaned_flights.csv: {flights.shape[0]:,} rows x {flights.shape[1]} columns")
except FileNotFoundError:
    raise FileNotFoundError(
        "Could not find 'cleaned_flights.csv'. Run 01_data_cleaning.ipynb first and upload the "
        "resulting file into this Colab session, or make sure both notebooks share the same runtime/session."
    )
except Exception as e:
    raise Exception(f"Unexpected error while loading cleaned_flights.csv: {e}")

if 'fl_date' in flights.columns:
    flights['fl_date'] = pd.to_datetime(flights['fl_date'], errors='coerce')

print("\nColumns available:")
print(list(flights.columns))


Loaded cleaned_flights.csv: 1,928,369 rows x 41 columns

Columns available:
['unnamed_0', 'year', 'month', 'dayofmonth', 'dayofweek', 'deptime', 'crsdeptime', 'arrtime', 'crsarrtime', 'uniquecarrier', 'flightnum', 'tailnum', 'actualelapsedtime', 'crselapsedtime', 'airtime', 'arrdelay', 'depdelay', 'origin', 'dest', 'distance', 'taxiin', 'taxiout', 'cancelled', 'cancellationcode', 'diverted', 'carrierdelay', 'weatherdelay', 'nasdelay', 'securitydelay', 'lateaircraftdelay', 'fl_date', 'day_of_week', 'crs_dep_hour', 'dep_hour', 'crs_arr_hour', 'arr_hour', 'dep_del15', 'carrier_type', 'prev_arr_delay', 'propagation_flag', 'had_weather_delay']


In [18]:
try:
    con = duckdb.connect(database=':memory:')
    con.register('flights', flights)
    print("DuckDB in-memory connection created and 'flights' DataFrame registered as a SQL table.")
except Exception as e:
    raise Exception(f"Failed to initialize DuckDB or register DataFrame: {e}")


DuckDB in-memory connection created and 'flights' DataFrame registered as a SQL table.


## 3. Basic Analyses

### 3.1 Total Flights, Average Delay, and Severe-Delay Rate by Airport

In [19]:
query_basic_otp = """
SELECT
    origin AS airport,
    COUNT(*) AS total_flights,
    ROUND(AVG(depdelay), 2) AS avg_dep_delay_min,
    ROUND(100.0 * SUM(CASE WHEN dep_del15 = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS severe_delay_rate_pct
FROM flights
GROUP BY origin
HAVING COUNT(*) >= 100
ORDER BY severe_delay_rate_pct DESC
"""

try:
    otp_by_airport = con.execute(query_basic_otp).df()
    print(query_basic_otp)
    display(otp_by_airport)
except Exception as e:
    raise Exception(f"Query failed (basic delay rate by airport): {e}")

print(f"\nKey finding: highest-severity airport is {otp_by_airport.iloc[0]['airport']} "
      f"at {otp_by_airport.iloc[0]['severe_delay_rate_pct']}% of its (already delayed) flights >=15 min late.")



SELECT
    origin AS airport,
    COUNT(*) AS total_flights,
    ROUND(AVG(depdelay), 2) AS avg_dep_delay_min,
    ROUND(100.0 * SUM(CASE WHEN dep_del15 = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS severe_delay_rate_pct
FROM flights
GROUP BY origin
HAVING COUNT(*) >= 100
ORDER BY severe_delay_rate_pct DESC



,airport,total_flights,avg_dep_delay_min,severe_delay_rate_pct
0,CEC,429,64.92,86.48
1,BGM,102,64.77,86.27
2,LMT,156,65.75,85.26
3,PMD,196,59.24,85.20
4,SPI,356,76.83,84.27
...,...,...,...,...
264,CLD,409,29.62,55.99
265,BQN,327,37.37,55.05
266,IPL,147,28.52,53.74
267,EKO,190,32.61,52.11



Key finding: highest-severity airport is CEC at 86.48% of its (already delayed) flights >=15 min late.


### 3.2 Delay Rate by Route (Origin -> Destination)

In [20]:
query_route_delay = """
SELECT
    origin,
    dest,
    COUNT(*) AS total_flights,
    ROUND(100.0 * SUM(dep_del15) / COUNT(*), 2) AS delay_rate_pct
FROM flights
GROUP BY origin, dest
HAVING COUNT(*) >= 30
ORDER BY delay_rate_pct DESC
LIMIT 20
"""

try:
    route_delay = con.execute(query_route_delay).df()
    print(query_route_delay)
    display(route_delay)
except Exception as e:
    raise Exception(f"Query failed (delay rate by route): {e}")

print(f"\nKey finding: worst route is {route_delay.iloc[0]['origin']} -> {route_delay.iloc[0]['dest']} "
      f"with a {route_delay.iloc[0]['delay_rate_pct']}% severe-delay rate.")



SELECT
    origin,
    dest,
    COUNT(*) AS total_flights,
    ROUND(100.0 * SUM(dep_del15) / COUNT(*), 2) AS delay_rate_pct
FROM flights
GROUP BY origin, dest
HAVING COUNT(*) >= 30
ORDER BY delay_rate_pct DESC
LIMIT 20



,origin,dest,total_flights,delay_rate_pct
0,ORF,JFK,96,93.75
1,BNA,BOS,30,93.33
2,HDN,ORD,97,92.78
3,JFK,BGR,52,92.31
4,BDL,JFK,111,91.89
5,JFK,IND,36,91.67
6,RDU,XNA,46,91.30
7,LMT,SFO,101,91.09
8,RIC,PHL,55,90.91
9,PHL,JFK,176,90.91



Key finding: worst route is ORF -> JFK with a 93.75% severe-delay rate.


### 3.3 Delay Cause Distribution

In [21]:
query_cause_dist = """
SELECT 'Carrier' AS delay_cause, ROUND(SUM(carrierdelay), 0) AS total_minutes FROM flights
UNION ALL
SELECT 'Weather', ROUND(SUM(weatherdelay), 0) FROM flights
UNION ALL
SELECT 'NAS (System)', ROUND(SUM(nasdelay), 0) FROM flights
UNION ALL
SELECT 'Security', ROUND(SUM(securitydelay), 0) FROM flights
UNION ALL
SELECT 'Late Aircraft', ROUND(SUM(lateaircraftdelay), 0) FROM flights
ORDER BY total_minutes DESC
"""

try:
    cause_dist = con.execute(query_cause_dist).df()
    cause_dist['pct_of_total'] = (cause_dist['total_minutes'] / cause_dist['total_minutes'].sum() * 100).round(2)
    print(query_cause_dist)
    display(cause_dist)
except Exception as e:
    raise Exception(f"Query failed (delay cause distribution): {e}")

print(f"\nKey finding: '{cause_dist.iloc[0]['delay_cause']}' is the largest contributor at "
      f"{cause_dist.iloc[0]['pct_of_total']}% of total delay-minutes.")



SELECT 'Carrier' AS delay_cause, ROUND(SUM(carrierdelay), 0) AS total_minutes FROM flights
UNION ALL
SELECT 'Weather', ROUND(SUM(weatherdelay), 0) FROM flights
UNION ALL
SELECT 'NAS (System)', ROUND(SUM(nasdelay), 0) FROM flights
UNION ALL
SELECT 'Security', ROUND(SUM(securitydelay), 0) FROM flights
UNION ALL
SELECT 'Late Aircraft', ROUND(SUM(lateaircraftdelay), 0) FROM flights
ORDER BY total_minutes DESC



,delay_cause,total_minutes,pct_of_total
0,Late Aircraft,31557038.0,39.97
1,Carrier,23926025.0,30.30
2,NAS (System),18739266.0,23.73
3,Weather,4620160.0,5.85
4,Security,112445.0,0.14



Key finding: 'Late Aircraft' is the largest contributor at 39.97% of total delay-minutes.


## 4. Intermediate Analyses

### 4.1 JOIN: Per-Airport Delay-Severity Tier Joined Back to Flights

A per-airport severity tier is built as a CTE and then **joined** back to the flights table so each individual flight carries its airport's overall tier -- useful for filtering/segmenting downstream analysis by how systemically troubled the origin airport is.

In [22]:
query_tier_join = """
WITH airport_tier AS (
    SELECT
        origin,
        CASE
            WHEN 100.0 * SUM(dep_del15) / COUNT(*) >= 70 THEN 'High Severity'
            WHEN 100.0 * SUM(dep_del15) / COUNT(*) >= 50 THEN 'Medium Severity'
            ELSE 'Lower Severity'
        END AS severity_tier
    FROM flights
    GROUP BY origin
    HAVING COUNT(*) >= 100
)
SELECT
    t.severity_tier,
    COUNT(*) AS total_flights,
    ROUND(AVG(f.depdelay), 2) AS avg_dep_delay_min,
    ROUND(100.0 * SUM(f.dep_del15) / COUNT(*), 2) AS delay_rate_pct
FROM flights f
JOIN airport_tier t
    ON f.origin = t.origin
GROUP BY t.severity_tier
ORDER BY delay_rate_pct DESC
"""

try:
    tier_join = con.execute(query_tier_join).df()
    print(query_tier_join)
    display(tier_join)
except Exception as e:
    raise Exception(f"Query failed (airport-tier JOIN): {e}")



WITH airport_tier AS (
    SELECT
        origin,
        CASE
            WHEN 100.0 * SUM(dep_del15) / COUNT(*) >= 70 THEN 'High Severity'
            WHEN 100.0 * SUM(dep_del15) / COUNT(*) >= 50 THEN 'Medium Severity'
            ELSE 'Lower Severity'
        END AS severity_tier
    FROM flights
    GROUP BY origin
    HAVING COUNT(*) >= 100
)
SELECT
    t.severity_tier,
    COUNT(*) AS total_flights,
    ROUND(AVG(f.depdelay), 2) AS avg_dep_delay_min,
    ROUND(100.0 * SUM(f.dep_del15) / COUNT(*), 2) AS delay_rate_pct
FROM flights f
JOIN airport_tier t
    ON f.origin = t.origin
GROUP BY t.severity_tier
ORDER BY delay_rate_pct DESC



,severity_tier,total_flights,avg_dep_delay_min,delay_rate_pct
0,High Severity,586704,47.91,73.91
1,Medium Severity,1340159,37.76,66.15


### 4.2 RANK(): Airports Ranked by Delay Rate

In [23]:
query_rank = """
SELECT
    origin AS airport,
    COUNT(*) AS total_flights,
    ROUND(100.0 * SUM(dep_del15) / COUNT(*), 2) AS delay_rate_pct,
    RANK() OVER (ORDER BY 100.0 * SUM(dep_del15) / COUNT(*) DESC) AS delay_rank
FROM flights
GROUP BY origin
HAVING COUNT(*) >= 100
ORDER BY delay_rank
"""

try:
    airport_rank = con.execute(query_rank).df()
    print(query_rank)
    display(airport_rank)
except Exception as e:
    raise Exception(f"Query failed (RANK airports by delay rate): {e}")

print(f"\nKey finding: top 3 highest-delay-rate airports are "
      f"{', '.join(airport_rank.head(3)['airport'].tolist())}.")



SELECT
    origin AS airport,
    COUNT(*) AS total_flights,
    ROUND(100.0 * SUM(dep_del15) / COUNT(*), 2) AS delay_rate_pct,
    RANK() OVER (ORDER BY 100.0 * SUM(dep_del15) / COUNT(*) DESC) AS delay_rank
FROM flights
GROUP BY origin
HAVING COUNT(*) >= 100
ORDER BY delay_rank



,airport,total_flights,delay_rate_pct,delay_rank
0,CEC,429,86.48,1
1,BGM,102,86.27,2
2,LMT,156,85.26,3
3,PMD,196,85.20,4
4,SPI,356,84.27,5
...,...,...,...,...
264,CLD,409,55.99,265
265,BQN,327,55.05,266
266,IPL,147,53.74,267
267,EKO,190,52.11,268



Key finding: top 3 highest-delay-rate airports are CEC, BGM, LMT.


### 4.3 CTE: Delay Summary Table

In [24]:
query_cte_summary = """
WITH delay_summary AS (
    SELECT
        origin AS airport,
        COUNT(*) AS total_flights,
        SUM(dep_del15) AS total_severe_delayed_flights,
        ROUND(AVG(depdelay), 2) AS avg_dep_delay_min,
        ROUND(AVG(carrierdelay), 2) AS avg_carrier_delay,
        ROUND(AVG(weatherdelay), 2) AS avg_weather_delay,
        ROUND(AVG(lateaircraftdelay), 2) AS avg_late_aircraft_delay
    FROM flights
    GROUP BY origin
    HAVING COUNT(*) >= 100
)
SELECT
    airport,
    total_flights,
    total_severe_delayed_flights,
    ROUND(100.0 * total_severe_delayed_flights / total_flights, 2) AS delay_rate_pct,
    avg_dep_delay_min,
    avg_carrier_delay,
    avg_weather_delay,
    avg_late_aircraft_delay
FROM delay_summary
ORDER BY delay_rate_pct DESC
"""

try:
    delay_summary_df = con.execute(query_cte_summary).df()
    print(query_cte_summary)
    display(delay_summary_df)
except Exception as e:
    raise Exception(f"Query failed (CTE delay summary): {e}")



WITH delay_summary AS (
    SELECT
        origin AS airport,
        COUNT(*) AS total_flights,
        SUM(dep_del15) AS total_severe_delayed_flights,
        ROUND(AVG(depdelay), 2) AS avg_dep_delay_min,
        ROUND(AVG(carrierdelay), 2) AS avg_carrier_delay,
        ROUND(AVG(weatherdelay), 2) AS avg_weather_delay,
        ROUND(AVG(lateaircraftdelay), 2) AS avg_late_aircraft_delay
    FROM flights
    GROUP BY origin
    HAVING COUNT(*) >= 100
)
SELECT
    airport,
    total_flights,
    total_severe_delayed_flights,
    ROUND(100.0 * total_severe_delayed_flights / total_flights, 2) AS delay_rate_pct,
    avg_dep_delay_min,
    avg_carrier_delay,
    avg_weather_delay,
    avg_late_aircraft_delay
FROM delay_summary
ORDER BY delay_rate_pct DESC



,airport,total_flights,total_severe_delayed_flights,delay_rate_pct,avg_dep_delay_min,avg_carrier_delay,avg_weather_delay,avg_late_aircraft_delay
0,CEC,429,371.0,86.48,64.92,10.07,1.83,21.43
1,BGM,102,88.0,86.27,64.77,17.90,7.11,34.10
2,LMT,156,133.0,85.26,65.75,15.49,1.78,23.84
3,PMD,196,167.0,85.20,59.24,8.06,0.00,15.34
4,SPI,356,300.0,84.27,76.83,18.70,0.71,22.62
...,...,...,...,...,...,...,...,...
264,CLD,409,229.0,55.99,29.62,7.92,0.58,13.23
265,BQN,327,180.0,55.05,37.37,15.04,0.00,17.28
266,IPL,147,79.0,53.74,28.52,10.70,2.30,11.29
267,EKO,190,99.0,52.11,32.61,12.00,0.28,16.96


## 5. Advanced Analyses — Delay Propagation

### 5.1 LAG(): Previous Flight's Arrival Delay for the Same Aircraft

The true aircraft-rotation signal: for each `tailnum`, what was the arrival delay of its immediately preceding flight.

In [25]:
query_lag = """
SELECT
    tailnum,
    fl_date,
    origin,
    dest,
    crsdeptime,
    depdelay,
    arrdelay,
    LAG(arrdelay) OVER (
        PARTITION BY tailnum ORDER BY fl_date, crsdeptime
    ) AS prev_flight_arr_delay
FROM flights
WHERE tailnum IS NOT NULL
ORDER BY tailnum, fl_date, crsdeptime
"""

try:
    lag_result = con.execute(query_lag).df()
    print(query_lag)
    display(lag_result.head(15))
    print(f"\n{lag_result['prev_flight_arr_delay'].notnull().sum():,} flights have a previous-leg arrival delay value.")
except Exception as e:
    raise Exception(f"Query failed (LAG previous arrival delay): {e}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


SELECT
    tailnum,
    fl_date,
    origin,
    dest,
    crsdeptime,
    depdelay,
    arrdelay,
    LAG(arrdelay) OVER (
        PARTITION BY tailnum ORDER BY fl_date, crsdeptime
    ) AS prev_flight_arr_delay
FROM flights
WHERE tailnum IS NOT NULL
ORDER BY tailnum, fl_date, crsdeptime



,tailnum,fl_date,origin,dest,crsdeptime,depdelay,arrdelay,prev_flight_arr_delay
0,80009E,2008-01-03,DTW,SBN,1204,95.0,95.0,NaN
1,80009E,2008-01-03,SBN,DTW,1328,96.0,96.0,95.0
2,80009E,2008-01-03,DTW,LNK,1515,80.0,63.0,96.0
3,80009E,2008-01-03,LNK,DTW,1655,53.0,38.0,63.0
4,80009E,2008-01-04,DTW,MLI,1922,23.0,7.0,38.0
5,80009E,2008-01-05,RIC,DTW,1235,12.0,26.0,7.0
6,80009E,2008-01-06,SAV,DTW,710,13.0,26.0,26.0
7,80009E,2008-01-06,DTW,BTV,1016,16.0,26.0,26.0
8,80009E,2008-01-06,BTV,DTW,1228,23.0,18.0,26.0
9,80009E,2008-01-06,DTW,LIT,1516,43.0,39.0,18.0



1,923,006 flights have a previous-leg arrival delay value.


### 5.2 CASE WHEN: Propagation Flag

In [26]:
query_prop_flag = """
WITH flight_sequence AS (
    SELECT
        tailnum,
        fl_date,
        origin,
        dest,
        uniquecarrier,
        depdelay,
        dep_del15,
        LAG(arrdelay) OVER (
            PARTITION BY tailnum ORDER BY fl_date, crsdeptime
        ) AS prev_flight_arr_delay
    FROM flights
    WHERE tailnum IS NOT NULL
)
SELECT
    *,
    CASE
        WHEN prev_flight_arr_delay >= 15 AND dep_del15 = 1 THEN 1
        ELSE 0
    END AS propagation_flag
FROM flight_sequence
"""

try:
    prop_flag_df = con.execute(query_prop_flag).df()
    display(prop_flag_df.head(10))
    print(f"\nTotal propagation events flagged: {prop_flag_df['propagation_flag'].sum():,} "
          f"({prop_flag_df['propagation_flag'].mean()*100:.2f}% of flights with a known previous leg)")
except Exception as e:
    raise Exception(f"Query failed (CASE WHEN propagation flag): {e}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tailnum,fl_date,origin,dest,uniquecarrier,depdelay,dep_del15,prev_flight_arr_delay,propagation_flag
0,80299E,2008-01-01,MSP,TUL,9E,176.0,1,NaN,0
1,80299E,2008-01-01,TUL,MSP,9E,176.0,1,182.0,1
2,80299E,2008-01-01,MSP,ICT,9E,124.0,1,182.0,1
3,80299E,2008-01-03,GRR,DTW,9E,26.0,1,112.0,1
4,80299E,2008-01-03,DTW,HSV,9E,20.0,1,26.0,1
5,80299E,2008-01-03,HSV,DTW,9E,13.0,0,22.0,0
6,80299E,2008-01-03,DTW,ELM,9E,8.0,0,24.0,0
7,80299E,2008-01-04,DTW,EVV,9E,38.0,1,-5.0,0
8,80299E,2008-01-04,EVV,DTW,9E,17.0,1,35.0,1
9,80299E,2008-01-06,DAY,MSP,9E,24.0,1,12.0,0



Total propagation events flagged: 958,376 (49.70% of flights with a known previous leg)


### 5.3 Propagation Score by Origin Airport

In [27]:
query_prop_score = """
WITH flight_sequence AS (
    SELECT
        tailnum,
        fl_date,
        origin,
        dep_del15,
        LAG(arrdelay) OVER (
            PARTITION BY tailnum ORDER BY fl_date, crsdeptime
        ) AS prev_flight_arr_delay
    FROM flights
    WHERE tailnum IS NOT NULL
),
flagged AS (
    SELECT
        *,
        CASE WHEN prev_flight_arr_delay >= 15 AND dep_del15 = 1 THEN 1 ELSE 0 END AS propagation_flag
    FROM flight_sequence
)
SELECT
    origin AS airport,
    COUNT(*) AS total_flights,
    SUM(propagation_flag) AS propagation_events,
    ROUND(1.0 * SUM(propagation_flag) / COUNT(*), 4) AS propagation_score
FROM flagged
GROUP BY origin
HAVING COUNT(*) >= 100
ORDER BY propagation_score DESC
"""

try:
    propagation_score_df = con.execute(query_prop_score).df()
    print(query_prop_score)
    display(propagation_score_df)
except Exception as e:
    raise Exception(f"Query failed (propagation score by airport): {e}")

print(f"\nKey finding: {propagation_score_df.iloc[0]['airport']} has the highest propagation score "
      f"({propagation_score_df.iloc[0]['propagation_score']}), making it the top candidate for "
      f"a source-airport intervention.")



WITH flight_sequence AS (
    SELECT
        tailnum,
        fl_date,
        origin,
        dep_del15,
        LAG(arrdelay) OVER (
            PARTITION BY tailnum ORDER BY fl_date, crsdeptime
        ) AS prev_flight_arr_delay
    FROM flights
    WHERE tailnum IS NOT NULL
),
flagged AS (
    SELECT
        *,
        CASE WHEN prev_flight_arr_delay >= 15 AND dep_del15 = 1 THEN 1 ELSE 0 END AS propagation_flag
    FROM flight_sequence
)
SELECT
    origin AS airport,
    COUNT(*) AS total_flights,
    SUM(propagation_flag) AS propagation_events,
    ROUND(1.0 * SUM(propagation_flag) / COUNT(*), 4) AS propagation_score
FROM flagged
GROUP BY origin
HAVING COUNT(*) >= 100
ORDER BY propagation_score DESC



,airport,total_flights,propagation_events,propagation_score
0,HHH,183,139.0,0.7596
1,SPI,356,266.0,0.7472
2,CEC,429,318.0,0.7413
3,CDV,212,152.0,0.7170
4,TRI,346,247.0,0.7139
...,...,...,...,...
264,DAL,18645,7579.0,0.4065
265,IAH,56571,22927.0,0.4053
266,ISP,2687,1079.0,0.4016
267,HLN,270,108.0,0.4000



Key finding: HHH has the highest propagation score (0.7596), making it the top candidate for a source-airport intervention.


### 5.4 Cohort Analysis: LCC vs FSC Propagation Patterns

In [28]:
query_cohort = """
WITH carrier_type AS (
    SELECT *,
        CASE
            WHEN uniquecarrier IN ('WN','NK','F9','FL') THEN 'LCC'
            WHEN uniquecarrier IN ('AA','DL','UA','US','NW','CO','HA','AS') THEN 'FSC'
            ELSE 'Other'
        END AS carrier_type
    FROM flights
),
flight_sequence AS (
    SELECT
        *,
        LAG(arrdelay) OVER (
            PARTITION BY tailnum ORDER BY fl_date, crsdeptime
        ) AS prev_flight_arr_delay
    FROM carrier_type
    WHERE tailnum IS NOT NULL
),
flagged AS (
    SELECT
        *,
        CASE WHEN prev_flight_arr_delay >= 15 AND dep_del15 = 1 THEN 1 ELSE 0 END AS propagation_flag
    FROM flight_sequence
)
SELECT
    carrier_type,
    COUNT(*) AS total_flights,
    ROUND(AVG(depdelay), 2) AS avg_dep_delay_min,
    SUM(propagation_flag) AS propagation_events,
    ROUND(1.0 * SUM(propagation_flag) / COUNT(*), 4) AS propagation_score
FROM flagged
WHERE carrier_type IN ('LCC', 'FSC')
GROUP BY carrier_type
ORDER BY propagation_score DESC
"""

try:
    cohort_df = con.execute(query_cohort).df()
    print(query_cohort)
    display(cohort_df)
except Exception as e:
    raise Exception(f"Query failed (LCC vs FSC cohort analysis): {e}")

if len(cohort_df) >= 2:
    print(f"\nKey finding: {cohort_df.iloc[0]['carrier_type']} carriers show a higher propagation score "
          f"({cohort_df.iloc[0]['propagation_score']}) than {cohort_df.iloc[1]['carrier_type']} "
          f"({cohort_df.iloc[1]['propagation_score']}).")



WITH carrier_type AS (
    SELECT *,
        CASE
            WHEN uniquecarrier IN ('WN','NK','F9','FL') THEN 'LCC'
            WHEN uniquecarrier IN ('AA','DL','UA','US','NW','CO','HA','AS') THEN 'FSC'
            ELSE 'Other'
        END AS carrier_type
    FROM flights
),
flight_sequence AS (
    SELECT
        *,
        LAG(arrdelay) OVER (
            PARTITION BY tailnum ORDER BY fl_date, crsdeptime
        ) AS prev_flight_arr_delay
    FROM carrier_type
    WHERE tailnum IS NOT NULL
),
flagged AS (
    SELECT
        *,
        CASE WHEN prev_flight_arr_delay >= 15 AND dep_del15 = 1 THEN 1 ELSE 0 END AS propagation_flag
    FROM flight_sequence
)
SELECT
    carrier_type,
    COUNT(*) AS total_flights,
    ROUND(AVG(depdelay), 2) AS avg_dep_delay_min,
    SUM(propagation_flag) AS propagation_events,
    ROUND(1.0 * SUM(propagation_flag) / COUNT(*), 4) AS propagation_score
FROM flagged
WHERE carrier_type IN ('LCC', 'FSC')
GROUP BY carrier_type
ORDER BY propagation_score DESC



,carrier_type,total_flights,avg_dep_delay_min,propagation_events,propagation_score
0,FSC,768605,40.93,369078.0,0.4802
1,LCC,475392,34.20,210244.0,0.4423



Key finding: FSC carriers show a higher propagation score (0.4802) than LCC (0.4423).


## 6. Export Final Propagation Analysis Dataset

In [31]:
query_export = """
WITH flight_sequence AS (
    SELECT
        *,
        LAG(arrdelay) OVER (
            PARTITION BY tailnum ORDER BY fl_date, crsdeptime
        ) AS prev_flight_arr_delay
    FROM flights
    WHERE tailnum IS NOT NULL
),
flagged AS (
    SELECT
        *,
        CASE WHEN prev_flight_arr_delay >= 15 AND dep_del15 = 1 THEN 1 ELSE 0 END AS propagation_flag,
        CASE
            WHEN uniquecarrier IN ('WN','NK','F9','FL') THEN 'LCC'
            WHEN uniquecarrier IN ('AA','DL','UA','US','NW','CO','HA','AS') THEN 'FSC'
            ELSE 'Other'
        END AS carrier_type
    FROM flight_sequence
)
SELECT
    origin AS airport,
    dest,
    uniquecarrier,
    carrier_type,
    fl_date,
    tailnum,
    depdelay,
    arrdelay,
    dep_del15,
    prev_flight_arr_delay,
    propagation_flag,
    carrierdelay,
    weatherdelay,
    nasdelay,
    securitydelay,
    lateaircraftdelay
FROM flagged
"""

try:
    export_df = con.execute(query_export).df()
    export_df.to_csv('airline_delay_analysis.csv', index=False)
    print(f"Exported 'airline_delay_analysis.csv': {export_df.shape[0]:,} rows x {export_df.shape[1]} columns")
except Exception as e:
    raise Exception(f"Failed to build/export final analysis dataset: {e}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported 'airline_delay_analysis.csv': 1,928,366 rows x 16 columns


## 7. Key Findings Summary

In [32]:
print("=" * 70)
print("SQL ANALYSIS — KEY FINDINGS SUMMARY")
print("=" * 70)
print(f"""
1. Highest-severity airport:  {otp_by_airport.iloc[0]['airport']} ({otp_by_airport.iloc[0]['severe_delay_rate_pct']}%)
2. Worst route by delay rate: {route_delay.iloc[0]['origin']} -> {route_delay.iloc[0]['dest']} ({route_delay.iloc[0]['delay_rate_pct']}%)
3. Largest delay-minute cause: {cause_dist.iloc[0]['delay_cause']} ({cause_dist.iloc[0]['pct_of_total']}% of total)
4. Highest-propagation-score airport: {propagation_score_df.iloc[0]['airport']} (score = {propagation_score_df.iloc[0]['propagation_score']})
5. LCC vs FSC propagation:    see cohort_df above

Final export: airline_delay_analysis.csv ({export_df.shape[0]:,} rows) — ready for Power BI.
""")


SQL ANALYSIS — KEY FINDINGS SUMMARY

1. Highest-severity airport:  CEC (86.48%)
2. Worst route by delay rate: ORF -> JFK (93.75%)
3. Largest delay-minute cause: Late Aircraft (39.97% of total)
4. Highest-propagation-score airport: HHH (score = 0.7596)
5. LCC vs FSC propagation:    see cohort_df above

Final export: airline_delay_analysis.csv (1,928,366 rows) — ready for Power BI.

